In [1]:
#QUESTO SCRIPT TOGLIE IL PRIMO MINUTO DI DATI X MUNGITURA, IPOTIZZANDO CHE LE PRIME LETTURE
#NON SI RIFERISCANO ALL ANIMALE PRESENTE MA A QUELLO PRECEDENTE. DOPODICHE FA UN ULTERIORE TAGLIO 
#ANDANDO A PRENDERE I SUCCESSIVI 240 SEC (4MIN) PER MUNGITURA

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
import os

In [2]:
path="/mnt/c/Users/isabe/OneDrive/Desktop/Lavoro/Tuscia/chiara/peaks_background/bufale_isa/"
os.chdir(path)

In [3]:
mandria_path = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_1/ch4_clean_out_bg_10.csv'
mandria = pd.read_csv(mandria_path)

In [4]:
mandria['ID_animal'] = mandria['ID_animal'].apply(lambda x: int(x))
mandria['Milking_number'] = mandria['Milking_number'].apply(lambda x: int(x))

In [5]:
mandria.tail()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Duration (hh:mm:ss),Milking_end,Milking_number
863081,2024-07-23 12:42:29.867926,126.456320,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
863082,2024-07-23 12:42:30.871007,101.456450,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
863083,2024-07-23 12:42:31.873548,78.356581,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
863084,2024-07-23 12:42:32.875246,59.856711,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740
863085,2024-07-23 12:45:00.238828,2.666536,2024-07-23,2024-07-23 12:45:00,3.73,245591,6.08,0 days 00:06:08,2024-07-23 12:51:08,1741


In [6]:
#dividing and plotting each milking (keeping animal information)
#inspection of milking with calculus of mean, median, standard deviation without removal of data

animals = list(set(mandria['ID_animal']))
mungiture = list(set(mandria['Milking_number']))
print(len(animals))
print(len(mungiture))

matrix= list()

mandria_cutted=mandria

36
1741


In [7]:
mandria_cutted.head()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Duration (hh:mm:ss),Milking_end,Milking_number
0,2024-07-01 09:54:00.472160,102.060696,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1
1,2024-07-01 09:54:01.474686,92.160826,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1
2,2024-07-01 09:54:02.476053,131.360957,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1
3,2024-07-01 09:54:03.478415,128.561087,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1
4,2024-07-01 09:54:04.481498,34.261218,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1


In [8]:
mandria_cutted['Milking_duration'] = mandria_cutted['Milking_duration'].apply(lambda x: str(x))


In [9]:
# Convert 'milking_duration' to seconds
mandria_cutted['milking_duration_seconds'] = mandria_cutted['Milking_duration'].apply(lambda x: int(x.split('.')[0]) * 60 + int(x.split('.')[1]))

#Assign a progressive number of seconds starting from 1 within each milking session
mandria_cutted['progressive_seconds'] = mandria_cutted.groupby(['ID_animal', 'Milking_number']).cumcount() + 1

# Filter the DataFrame to keep only records after the first minute (60 seconds)
mandria_cutted = mandria_cutted[mandria_cutted['milking_duration_seconds'] >= 60]

print(mandria_cutted.shape)
print(len(set(mandria_cutted['Milking_number'])))

# Filter the DataFrame to keep only records of milkings longer than 200 seconds
filtered_df = mandria_cutted[mandria_cutted['milking_duration_seconds'] >= 240]

print(filtered_df.shape)
print(len(set(filtered_df['Milking_number'])))
print(len(set(filtered_df['ID_animal'])))


(863086, 12)
1741
(862880, 12)
1740
36


In [10]:
#taglio a 4 minuti per mungitura (240+60=300)
filtered_df2 = filtered_df[filtered_df['progressive_seconds'] <= 300]

In [11]:
filtered_df2.shape

(520515, 12)

In [12]:
print(len(set(filtered_df2['Milking_number'])))
print(len(set(filtered_df2['ID_animal'])))


1740
36


In [13]:
ch4_mean = filtered_df2['CH4'].mean()
print(ch4_mean)

# Salva il DataFrame combinato in un nuovo file CSV.
#mandria_path = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/ch4_clean_out_bg_10.csv'

output_file = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_1/ch4_cutted_milkings.csv'
filtered_df2.to_csv(output_file, index=False)




143.40692494312216


In [14]:
ch4_cutted_milkings = './dati_bufale/dati_grezzi_agosto_luglio/Dataset_CH4/Bufale_Rinna/Luglio/protocollo_1/ch4_cutted_milkings.csv'
ch4_cut_milkings = pd.read_csv(ch4_cutted_milkings)
ch4_cut_milkings.head()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Duration (hh:mm:ss),Milking_end,Milking_number,milking_duration_seconds,progressive_seconds
0,2024-07-01 09:54:00.472160,102.060696,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1,567,1
1,2024-07-01 09:54:01.474686,92.160826,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1,567,2
2,2024-07-01 09:54:02.476053,131.360957,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1,567,3
3,2024-07-01 09:54:03.478415,128.561087,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1,567,4
4,2024-07-01 09:54:04.481498,34.261218,2024-07-01,2024-07-01 09:54:00,1.44,5983,9.27,0 days 00:09:27,2024-07-01 10:03:27,1,567,5


In [15]:
ch4_cut_milkings.tail()

,timestamp,CH4,date,Ora inizio,Produzione (kg),ID_animal,Milking_duration,Duration (hh:mm:ss),Milking_end,Milking_number,milking_duration_seconds,progressive_seconds
520510,2024-07-23 12:42:29.867926,126.456320,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740,273,261
520511,2024-07-23 12:42:30.871007,101.456450,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740,273,262
520512,2024-07-23 12:42:31.873548,78.356581,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740,273,263
520513,2024-07-23 12:42:32.875246,59.856711,2024-07-23,2024-07-23 12:38:00,3.07,233733,4.33,0 days 00:04:33,2024-07-23 12:42:33,1740,273,264
520514,2024-07-23 12:45:00.238828,2.666536,2024-07-23,2024-07-23 12:45:00,3.73,245591,6.08,0 days 00:06:08,2024-07-23 12:51:08,1741,368,1
